# FineBERT

In [ ]:
import torch
import transformers

if torch.cuda.is_available():
    print( {torch.cuda.get_device_name(0)})

### Pipeline Finebert model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import numpy as np

tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")

print(model)
print(model.config.id2label)
print(tokenizer.vocab_size)



In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [ ]:
model.eval()

In [ ]:
import pandas as pd

train_df = pd.read_csv('../data/processed/train.csv')
valid_df = pd.read_csv('../data/processed/valid.csv')
test_df = pd.read_csv('../data/processed/test.csv')

In [ ]:
X_test_raw = test_df['sentence']
X_test = test_df['clean_text']
y_test = test_df['sentiment']

In [ ]:
from transformers import pipeline

finbert = pipeline(
    "sentiment-analysis",
    model = "ProsusAI/finbert",
    tokenizer = "ProsusAI/finbert"

)

In [ ]:
zero_shot_results = finbert(list(X_test_raw), batch_size=32)
y_pred = [r['label'].lower() for r in zero_shot_results]

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

### Manual model

In [ ]:
train_df = pd.read_csv('../data/processed/train.csv')
valid_df = pd.read_csv('../data/processed/valid.csv')
test_df = pd.read_csv('../data/processed/test.csv')

In [ ]:
X_train = train_df['clean_text']
y_train = train_df['sentiment']

X_val = valid_df['clean_text']
y_val = valid_df['sentiment']

X_test = test_df['clean_text']
y_test = test_df['sentiment']

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_val = le.transform(y_val)
y_test = le.transform(y_test)

In [ ]:
class CustomDataset:

    def __init__(self, text, labels, tokenizer, max_len = 128):
        self.text = text    
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.text)
    
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.text[idx],
            max_length=128,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
            ) 
        return {
            'input_ids' : encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [ ]:
train_dataset = CustomDataset(
    text = train_df['sentence'].tolist(),
    labels=y_train,
    tokenizer=tokenizer)

valid_dataset = CustomDataset(
    text=valid_df['sentence'].tolist(),
    labels=y_val,
    tokenizer=tokenizer)

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")
model.eval()

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr = 2e-5, weight_decay=0.01)

In [ ]:
from transformers import get_linear_schedule_with_warmup

num_epochs = 3
training_steps = len(train_loader) * num_epochs
warmup_steps = int(training_steps * 0.1)

scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_training_steps=training_steps,
    num_warmup_steps=warmup_steps
)

In [ ]:
from torch.nn import utils

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0

    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        label = batch['label'].to(device)
        optimizer.zero_grad()

        outputs = model(
            input_ids = input_ids,
            labels = label,
            attention_mask = attention_mask
        )

        loss = outputs.loss

        loss.backward()
        max_norm = 1.0
        
        utils.clip_grad_norm_(model.parameters(),
                              max_norm = 1.0)

        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        
    return total_loss/len(loader)